# Read DDH file (dta)

In [ ]:
def DDHReadDta(i_ddh=0,name_ddh='FCTPRECCSCOL',
                                                    i_ech=198,npdt=2,direction=-1,
                                                    n_lon=170, n_lat=84, n_pre=90,dt_model=0.125):
    # constante pour remetre les K/j en K donc /8 (npdt_h) car on a 8*dt (=15min/2) en 1h, 
    # comme on a 24h dans une journée, on fait donc /24 et 
    # on fait /npdt(=2 ) pour remettre en Delta t et pas dt.]
    ccc= 2* 24 / dt_model  
    

    # Read at t and t +/- dt_traj
    DDHt=np.loadtxt('/home/wimmerm/NAWDEX/Data/fc_000/DHFDLARPE+'+str(i_ech).zfill(4)+".ren.manq.tmp."+name_ddh+".dta")
    DDHt1=np.loadtxt('/home/wimmerm/NAWDEX/Data/fc_000/DHFDLARPE+'+str(i_ech-npdt).zfill(4)+".ren.manq.tmp."+name_ddh+".dta")

    VAR_ddh= (DDHt[:,3]*i_ech - DDHt1[:,3]*(i_ech-npdt)) / ccc #(npdt * ccc)

    LON_ddh=(DDHt[:,0] + 180) % 360 - 180   
    LAT_ddh=DDHt[:,1]
    PRE_ddh=DDHt[:,2]*(-100)
    
    Point_to_delete=(LAT_ddh>90)+(PRE_ddh<0.5)  
    LON_ddh[Point_to_delete]=np.nan
    LAT_ddh[Point_to_delete]=np.nan        
    PRE_ddh[Point_to_delete]=np.nan 
    
    
    # 1D to 3D array 
    VAR_ddh =    np.transpose(VAR_ddh.reshape((n_lon,n_lat,n_pre)),(2,1,0))[:,::-1,:]
    LON_3D_ddh = np.transpose(LON_ddh.reshape((n_lon,n_lat,n_pre)),(2,1,0))[:,::-1,:]
    LAT_3D_ddh = np.transpose(LAT_ddh.reshape((n_lon,n_lat,n_pre)),(2,1,0))[:,::-1,:]
    PRE_3D_ddh = np.transpose(PRE_ddh.reshape((n_lon,n_lat,n_pre)),(2,1,0))[:,::-1,:]
    
    return VAR_ddh, LON_3D_ddh, LAT_3D_ddh, PRE_3D_ddh

# Fill missing points (over Greenland)

In [ ]:
def DDHMissingPointsPrev(DDH,n_lon, n_lat, n_pre):
    print(' - Missing Points')
    list_field=['PRESSURE']+list_DDH
    
    # Interpolation des points manquants
    if np.any(DDH['LATITUDE']>90) or np.any(DDH['PRESSURE']<0.5):
        position1=[]
        position2=[]
        if np.any(DDH['LATITUDE']>90) :
            print(' \t - Interpolation des points manquants LAT')
            position1 = np.where( DDH['LATITUDE'][0,:,:]>90 ) # position des points manquants

        elif np.any(DDH['PRESSURE']<0.5):
            print(' \t - Interpolation des points manquants PRE')
            position2 = np.where( DDH['PRESSURE'][:,0,0]<0.5) # position des points manquants

        for position in  [position1]:#,position2]:
          for i_point in range(0,len(position[0])):
            ilat=position[0][i_point]
            ilon=position[1][i_point]

                    
            if ilon ==n_lon-1  or  ilon==0:
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat+1,ilon]+
                                              DDH['LATITUDE'][:,ilat-1,ilon])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat+1,ilon]+
                                               DDH['LONGITUDE'][:,ilat-1,ilon])/2
                if ilon ==n_lon-1:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat+1,ilon]+
                                                 DDH[field][:,ilat-1,ilon])/3
                elif ilon==0:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat+1,ilon]+
                                                 DDH[field][:,ilat-1,ilon])/3

            if ilon ==n_lat-1  or  ilat==0:
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat,ilon+1]+
                                              DDH['LATITUDE'][:,ilat,ilon-1])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat,ilon+1]+
                                               DDH['LONGITUDE'][:,ilat,ilon-1])/2
                    
                if ilon ==n_lat-1:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat-1,ilon])/3
                elif ilat==0:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat+1,ilon])/3
            else :
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat+1,ilon]+
                                              DDH['LATITUDE'][:,ilat-1,ilon])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat,ilon+1]+
                                               DDH['LONGITUDE'][:,ilat,ilon-1])/2
                for field in list_field: 
                    DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                             DDH[field][:,ilat,ilon-1]+
                                             DDH[field][:,ilat+1,ilon]+
                                             DDH[field][:,ilat-1,ilon])/4
                        
    return DDH

In [ ]:
def DDHMissingPoints(DDH,n_lon, n_lat, n_pre):
    print(' - Missing Points')
    list_field=['PRESSURE']+list_DDH
    
    # Interpolation des points manquants
    Point_to_fill=np.where(np.isnan(VAR_DDH['LATITUDE'][0,:,:]))
    for ilat,ilon in zip(Point_to_fill[0],Point_to_fill[1]):                   
            if ilon ==n_lon-1  or  ilon==0:
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat+1,ilon]+
                                              DDH['LATITUDE'][:,ilat-1,ilon])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat+1,ilon]+
                                               DDH['LONGITUDE'][:,ilat-1,ilon])/2
                if ilon == n_lon-1:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat+1,ilon]+
                                                 DDH[field][:,ilat-1,ilon])/3
                elif ilon==0:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat+1,ilon]+
                                                 DDH[field][:,ilat-1,ilon])/3

            if ilat ==n_lat-1  or  ilat==0:
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat,ilon+1]+
                                              DDH['LATITUDE'][:,ilat,ilon-1])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat,ilon+1]+
                                               DDH['LONGITUDE'][:,ilat,ilon-1])/2
                    
                if ilat ==n_lat-1:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat-1,ilon])/3
                elif ilat==0:
                    for field in list_field: 
                        DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                                 DDH[field][:,ilat,ilon-1]+
                                                 DDH[field][:,ilat+1,ilon])/3
            else :
                DDH['LATITUDE'][:,ilat,ilon]=(DDH['LATITUDE'][:,ilat+1,ilon]+
                                              DDH['LATITUDE'][:,ilat-1,ilon])/2
                DDH['LONGITUDE'][:,ilat,ilon]=(DDH['LONGITUDE'][:,ilat,ilon+1]+
                                               DDH['LONGITUDE'][:,ilat,ilon-1])/2
                for field in list_field: 
                    DDH[field][:,ilat,ilon]=(DDH[field][:,ilat,ilon+1]+
                                             DDH[field][:,ilat,ilon-1]+
                                             DDH[field][:,ilat+1,ilon]+
                                             DDH[field][:,ilat-1,ilon])/4
                        
    return DDH

# Vertical smooth 

In [ ]:
def DDHSmoothing(DDH, list_DDH, type_Lissage):
    print(' - Smoothing')
    if type_Lissage=='Philippe':
        coeff=[0.06,0.1,0.2,0.5]
    elif type_Lissage=='Gwendal':
        coeff=[0.0002,0.0019,0.0110,0.0430,0.1142,0.2051,0.2493]
    elif type_Lissage=='Gwendal2':
        coeff=[0.0833,0.1429,0.1786,0.1905]        
    elif type_Lissage=='Gwendal3':        
        coeff=[0.0385,0.0699,0.0944,0.1119,0.1224,0.1259]
    elif type_Lissage=='Merde':
        coeff=[0.5, 0.25]
    
    extension='_'+type_Lissage
        
    nb_coeff=len(coeff)
    poids=sum(coeff+coeff[:-1][::-1])
    coeff_list=list(range(nb_coeff)) + list(range(nb_coeff-2, -1, -1))
    for field in list_DDH:
        tmp=np.array(DDH[field].data)
        tmp[nb_coeff-1:len(DDH['PRESSURE'])-nb_coeff,:,:]=0
        for j,i in enumerate(coeff_list):
            tmp[nb_coeff-1:len(DDH['PRESSURE'])-nb_coeff,:,:]+= coeff[i] * np.array(DDH[field][j:len(DDH['PRESSURE'])-(2*nb_coeff)+1+j,:,:])
        tmp[nb_coeff-1:len(DDH['PRESSURE']),:,:]/=poids
        DDH[field][:]=tmp
        
    return DDH, extension



# Transform T tendencies in Theta tendencies

In [ ]:
def DDHTheta(DDH,list_DDH,PRE_DDH):
    print(' - Theta')
    for field in list_DDH:
        tmp=np.array(DDH[field].data)
        tmp=tmp*(100000.0/PRE_DDH )**(ZR//cp)              #6332 #ZKP
        DDH[field][:]=tmp
    return DDH



# Interpolation on regular grid (same as in grib file)

In [ ]:
def DDHRegular(LON_model, LAT_model, PRE_model, 
              LON_3D_DDH, LAT_3D_DDH, PRE_3D_DDH,
              DDH, list_DDH):
    print(' - Interpolation on regular grid')
    
    LON_2D_DDH_new, LAT_2D_DDH_new=np.meshgrid(LON_model, LAT_model)
    
    shape_new_DDH= (PRE_3D_DDH.shape[0],)+ LON_2D_DDH_new.shape
    
    shape_1D_DDH= 1
    for i in LON_3D_DDH[0,:,:].shape:
        shape_1D_DDH=shape_1D_DDH*i

    LON_1D_DDH=LON_3D_DDH[0,:,:].reshape(shape_1D_DDH)
    LAT_1D_DDH=LAT_3D_DDH[0,:,:].reshape(shape_1D_DDH)

    DDH_final={}
    for i_ddh in list_DDH:
        DDH_final[i_ddh]=np.zeros((shape_new_DDH))
        for i_lev in range(PRE_3D_DDH.shape[0]):
            tmp = DDH[i_ddh][i_lev,:,:].reshape(shape_1D_DDH)
            DDH_final[i_ddh][i_lev,:,:] = griddata((LAT_1D_DDH, LON_1D_DDH), tmp, (LAT_2D_DDH_new, LON_2D_DDH_new), method='linear')

    PRE_3D_DDH_new=np.zeros((shape_new_DDH))
    for i_lev in range(PRE_3D_DDH.shape[0]):
        tmp = PRE_3D_DDH[i_lev,:,:].reshape(shape_1D_DDH)
        PRE_3D_DDH_new[i_lev,:,:] = griddata((LAT_1D_DDH, LON_1D_DDH), tmp, (LAT_2D_DDH_new, LON_2D_DDH_new), method='linear')

    return DDH_final, LON_2D_DDH_new, LAT_2D_DDH_new, PRE_3D_DDH_new


# Calculation of Gradient for PV framework

In [ ]:
def DDHGradLon(DDH_field, LAT, LON):
    nPRE,nLAT,nLON=DDH_field.shape
    
    dLON=float(LON[0,2]-LON[0,0])
    
    D=Ra*np.pi/180
    C=np.cos(LAT*np.pi/180)*D
    
    field_grad=np.zeros((DDH_field.shape))
    for i_lev in range(nPRE):
        field_grad[i_lev, 1:nLAT-1,1:nLON-1]=(DDH_field[i_lev,1:nLAT-1,2:nLON]-
                                              DDH_field[i_lev,1:nLAT-1,0:nLON-2])/(
                                              C[1:nLAT-1,1:nLON-1]*  dLON) # (LON[1:nLAT-1,2:nLON]-LON[1:nLAT-1,0:nLON-2]))
    return field_grad
                
def DDHGradLat(DDH_field, LAT, LON):
    nPRE,nLAT,nLON=DDH_field.shape
    
    dLAT=float(LAT[2,0]-LAT[0,0])
    
    D=Ra*np.pi/180

    field_grad=np.zeros((DDH_field.shape))
    for i_lev in range(nPRE):
        field_grad[i_lev, 1:nLAT-1,1:nLON-1]=(DDH_field[i_lev,2:nLAT,1:nLON-1]-
                                              DDH_field[i_lev,0:nLAT-2,1:nLON-1]) / (D*dLAT) #(LAT[2:nLAT,1:nLON-1]-LAT[0:nLAT-2,1:nLON-1]) *D)
        
    return field_grad           

def DDHGradPre(DDH_field,PRE):
    nPRE,nLAT,nLON=DDH_field.shape
    
    field_grad=np.zeros((DDH_field.shape))
    for i_lev in range(1,nPRE-1):
        field_grad[i_lev,:,:]=(DDH_field[i_lev+1,:,:]-DDH_field[i_lev-1,:,:])/(PRE[i_lev+1,:,:]-PRE[i_lev-1,:,:])
    return field_grad

def DDHGradIsobarLon(DDH_field, PRE, LAT, LON):
    nPRE,nLAT,nLON=DDH_field.shape

    D=Ra*np.pi/180
    C=np.cos(LAT*np.pi/180)*D
    
    field_grad=np.zeros((DDH_field.shape))
    for i_lev in range(1,nPRE-1):
        field_grad[i_lev,1:nLAT-1,1:nLON-1]=DDHGradLon(DDH_field, LAT, LON)[i_lev,1:nLAT-1,1:nLON-1] +(DDHGradPre(DDH_field, PRE)[i_lev,1:nLAT-1,1:nLON-1]*((PRE[i_lev,1:nLAT-1,2:nLON]-PRE[i_lev,1:nLAT-1,0:nLON-2])/((LON[1:nLAT-1,2:nLON]-LON[1:nLAT-1,0:nLON-2])*C[1:nLAT-1,1:nLON-1])))
    return field_grad        

def DDHGradIsobarLat(DDH_field, PRE, LAT, LON):
    nPRE,nLAT,nLON=DDH_field.shape

    D=Ra*np.pi/180
    
    field_grad=np.zeros((DDH_field.shape))
    for i_lev in range(1,nPRE-1):
        field_grad[i_lev,1:nLAT-1,1:nLON-1]=DDHGradLat(DDH_field, LAT, LON)[i_lev,1:nLAT-1,1:nLON-1] + (DDHGradPre(DDH_field, PRE)[i_lev,1:nLAT-1,1:nLON-1]*((PRE[i_lev,2:nLAT,1:nLON-1]-PRE[i_lev,0:nLAT-2,1:nLON-1])/((LAT[2:nLAT,1:nLON-1]-LAT[0:nLAT-2,1:nLON-1])*D)))
    return field_grad        

In [ ]:
def DDHGrad(DDH_f, PRE_DDH_reg, LAT_DDH_reg, LON_DDH_reg):
    # Compute Grad of tendencies
    DDH_grad={}
    for i_ddh in DDH_f.keys():
        print(i_ddh)
        DDH_grad['d'+i_ddh+'dx_P']=DDHGradIsobarLon(DDH_f[i_ddh], PRE_DDH_reg, LAT_DDH_reg, LON_DDH_reg)
        DDH_grad['d'+i_ddh+'dy_P']=DDHGradIsobarLat(DDH_f[i_ddh], PRE_DDH_reg, LAT_DDH_reg, LON_DDH_reg)
        DDH_grad['d'+i_ddh+'dP']  =DDHGradPre(DDH_f[i_ddh], PRE_DDH_reg)

    # Add Grad in DDH
    return DDH_f.update(DDH_grad)


# Add summed variables (on u, v, T)

In [ ]:
def DDHAddVar(DDH):
    print(' - Add summed variables')
    list_DDH_Heat=['FCTPRECCSCOL','FCTPRECCSCON',
          'FCTPRECCSSTL','FCTPRECCSSTN',
          'FCTPRECICOL','FCTPRECICON',
          'FCTPRECISTL','FCTPRECISTN',
          'FCTRAYSOL1','FCTRAYTER1',
          'FCTTUR','FCTTURCONV']
    list_DDH_U=['FUUTUR',"FUUTURCONV"]
    list_DDH_V=["FVVTUR","FVVTURCONV"] 
    
    DDH['chauf']=np.zeros((DDH['FCTPRECCSCOL'].shape))
    for i_ddh in list_DDH_Heat:
        DDH['chauf'] += DDH[i_ddh]
    
    DDH['accu']=np.zeros((DDH['FCTPRECCSCOL'].shape))
    for i_ddh in list_DDH_U:
        DDH['accu'] += DDH[i_ddh]    
        
    DDH['accv']=np.zeros((DDH['FCTPRECCSCOL'].shape))
    for i_ddh in list_DDH_V:
        DDH['accv'] += DDH[i_ddh] 
    return DDH

# Transform dictionnary to xarray

In [26]:
def Dict2xarray(list_DDH_name, DDH, PR, LA, LO,
               LON_xarray, LAT_xarray, PRE_xarray,dims=('lat','lon')):   
    
    Dict_for_xarray={}
    coord= {
            'pressure':PR,
            'latitude':(dims, LA),
            'longitude':(dims, LO),
                }
    for name_ddh in list_DDH_name:
        Dict_for_xarray[name_ddh]=(('pressure',)+dims, DDH[name_ddh])
        
    Dict_for_xarray['LONGITUDE']=(('pressure',)+dims, LON_xarray)
    Dict_for_xarray['LATITUDE']=(('pressure',)+dims, LAT_xarray)
    Dict_for_xarray['PRESSURE']=(('pressure',)+dims, PRE_xarray)           

    return xr.Dataset(data_vars=Dict_for_xarray,coords=coord)

In [ ]:

list_DDH=['chauf','accu','accv'
'dchaufdx_P','dchaufdy_P','dchaufdP',
'daccvdx_P','daccudy_P',
'daccudP','daccvdP',
'FCTPRECCSCOL','dFCTPRECCSCOLdx_P',
'dFCTPRECCSCOLdy_P', 'dFCTPRECCSCOLdP',
'FCTPRECCSCON','dFCTPRECCSCONdx_P',
'dFCTPRECCSCONdy_P', 'dFCTPRECCSCONdP',
'FCTPRECCSSTL','dFCTPRECCSSTLdx_P',
'dFCTPRECCSSTLdy_P', 'dFCTPRECCSSTLdP',
'FCTPRECCSSTN','dFCTPRECCSSTNdx_P',
'dFCTPRECCSSTNdy_P', 'dFCTPRECCSSTNdP',
'FCTPRECICOL','dFCTPRECICOLdx_P',
'dFCTPRECICOLdy_P', 'dFCTPRECICOLdP',
'FCTPRECICON','dFCTPRECICONdx_P',
'dFCTPRECICONdy_P', 'dFCTPRECICONdP',
'FCTPRECISTL','dFCTPRECISTLdx_P',
'dFCTPRECISTLdy_P', 'dFCTPRECISTLdP',
'FCTPRECISTN','dFCTPRECISTNdx_P',
'dFCTPRECISTNdy_P', 'dFCTPRECISTNdP',
'FCTRAYSOL1','dFCTRAYSOL1dx_P',
'dFCTRAYSOL1dy_P', 'dFCTRAYSOL1dP',
'FCTRAYTER1','dFCTRAYTER1dx_P',
'dFCTRAYTER1dy_P', 'dFCTRAYTER1dP',
'FCTTUR','dFCTTURdx_P',
'dFCTTURdy_P', 'dFCTTURdP',
'FCTTURCONV','dFCTTURCONVdx_P',
'dFCTTURCONVdy_P', 'dFCTTURCONVdP',
'FUUTUR','dFUUTURdy_P',
'FUUTURCONV','dFUUTURCONVdy_P',
'FVVTUR','dFVVTURdx_P',
'FVVTURCONV','dFVVTURCONVdx_P']


import xarray as xr
import pickle

def DDHReadNc(root, file_name, list_DDH):

    with open(root+file_name, 'rb') as ddh: 
        [lon2,lat2,pr_ddh,x_champ,y_champ,PR_champ,
                 CHAUF,TU,TV,CHAUFX,CHAUFY,CHAUFZ,CHAUFX_V,CHAUFY_U,CHAUFZ_U,CHAUFZ_V,
                 CHAUF_1,CHAUFGX_1,CHAUFGY_1,CHAUFGZ_1,CHAUF_2,CHAUFGX_2,CHAUFGY_2,CHAUFGZ_2,
                 CHAUF_3,CHAUFGX_3,CHAUFGY_3,CHAUFGZ_3,CHAUF_4,CHAUFGX_4,CHAUFGY_4,CHAUFGZ_4,
                 CHAUF_5,CHAUFGX_5,CHAUFGY_5,CHAUFGZ_5,CHAUF_6,CHAUFGX_6,CHAUFGY_6,CHAUFGZ_6,
                 CHAUF_7,CHAUFGX_7,CHAUFGY_7,CHAUFGZ_7,CHAUF_8,CHAUFGX_8,CHAUFGY_8,CHAUFGZ_8,
                 CHAUF_9,CHAUFGX_9,CHAUFGY_9,CHAUFGZ_9,CHAUF_10,CHAUFGX_10,CHAUFGY_10,CHAUFGZ_10,
                 CHAUF_11,CHAUFGX_11,CHAUFGY_11,CHAUFGZ_11,CHAUF_12,CHAUFGX_12,CHAUFGY_12,CHAUFGZ_12,
                 CHAUF_13,CHAUFGY_13,CHAUF_14,CHAUFGY_14,CHAUF_15,CHAUFGX_15,CHAUF_16,CHAUFGX_16]=pickle.load(ddh, encoding='latin1')
    ddh.close() 

    list_numpy=[CHAUF,TU,TV,CHAUFX,CHAUFY,CHAUFZ,CHAUFX_V,CHAUFY_U,CHAUFZ_U,CHAUFZ_V,
                 CHAUF_1,CHAUFGX_1,CHAUFGY_1,CHAUFGZ_1,CHAUF_2,CHAUFGX_2,CHAUFGY_2,CHAUFGZ_2,
                 CHAUF_3,CHAUFGX_3,CHAUFGY_3,CHAUFGZ_3,CHAUF_4,CHAUFGX_4,CHAUFGY_4,CHAUFGZ_4,
                 CHAUF_5,CHAUFGX_5,CHAUFGY_5,CHAUFGZ_5,CHAUF_6,CHAUFGX_6,CHAUFGY_6,CHAUFGZ_6,
                 CHAUF_7,CHAUFGX_7,CHAUFGY_7,CHAUFGZ_7,CHAUF_8,CHAUFGX_8,CHAUFGY_8,CHAUFGZ_8,
                 CHAUF_9,CHAUFGX_9,CHAUFGY_9,CHAUFGZ_9,CHAUF_10,CHAUFGX_10,CHAUFGY_10,CHAUFGZ_10,
                 CHAUF_11,CHAUFGX_11,CHAUFGY_11,CHAUFGZ_11,CHAUF_12,CHAUFGX_12,CHAUFGY_12,CHAUFGZ_12,
                 CHAUF_13,CHAUFGY_13,CHAUF_14,CHAUFGY_14,CHAUF_15,CHAUFGX_15,CHAUF_16,CHAUFGX_16]
    DDH={}
    for name_ddh, i_ddh in zip(list_DDH, list_numpy):
        DDH[name_ddh]=i_ddh.T
    
        
    y_champ, x_champ, _ = np.meshgrid(lat2, lon2, pr_ddh)
    y_champ=np.transpose(y_champ, (2,1,0))
    x_champ=np.transpose(x_champ, (2,1,0))
    PR_champ=np.transpose(PR_champ, (2,1,0))
    
    
    DDH=Dict2xarray(list_DDH, DDH, PR_champ[:,10,10], y_champ[1,:,:], x_champ[1,:,:],
               x_champ, y_champ, PR_champ ,dims=('lat','lon'))
    return DDH

DDH=DDHReadNc('/home/wimmerm/NAWDEX/Data/fc_000/', f'DDH_MW_0.5_198_retro.pkl',list_DDH)




In [ ]:
# Verif DDH ok